<a href="https://colab.research.google.com/github/kimmy111-zhu/human-validation/blob/main/notebooks/mmlu_human_validation_pipeline.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [16]:
import json
import pandas as pd
from google.colab import files


# =====================================
# Settings
# =====================================

INPUT_FILE = "/content/mmlu_rate_0.1.jsonl"

OUTPUT_CSV = "/content/mmlu_original_reference_full.csv"
OUTPUT_JSONL = "/content/mmlu_original_reference_full.jsonl"


# =====================================
# Extract all original MMLU records
# Important: do NOT remove duplicates
# =====================================

original_records = []

with open(INPUT_FILE, "r", encoding="utf-8-sig") as file:

    for line_number, line in enumerate(file, start=1):

        line = line.strip()

        if not line:
            continue

        try:
            record = json.loads(line)

        except json.JSONDecodeError as error:
            print(
                f"Warning: JSON error on line {line_number}: "
                f"{error}"
            )
            continue

        original_prompt = str(
            record.get("original_text_backup", "")
        ).strip()

        subject = str(
            record.get("subject", "")
        ).strip()

        choices = record.get("choices", [])

        gold_answer = record.get("answer", None)

        if not original_prompt:
            print(
                f"Warning: missing original_text_backup "
                f"on line {line_number}"
            )
            continue

        original_records.append(
            {
                "Original_ID": (
                    f"MMLU_{len(original_records) + 1:05d}"
                ),
                "Original_Row": line_number,
                "Original_Subject": subject,
                "Original_Prompt": original_prompt,
                "Original_Choices": json.dumps(
                    choices,
                    ensure_ascii=False
                ),
                "Gold_Answer": gold_answer,
            }
        )


# =====================================
# Create dataframe
# =====================================

original_df = pd.DataFrame(original_records)


# Count repeated prompts, but keep them
repeated_prompt_count = original_df.duplicated(
    subset=["Original_Prompt"]
).sum()

repeated_full_record_count = original_df.duplicated(
    subset=[
        "Original_Subject",
        "Original_Prompt",
        "Original_Choices",
        "Gold_Answer",
    ]
).sum()


# =====================================
# Save outputs
# =====================================

original_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig"
)

original_df.to_json(
    OUTPUT_JSONL,
    orient="records",
    lines=True,
    force_ascii=False
)


# =====================================
# Report
# =====================================

print("=" * 60)
print("MMLU ORIGINAL EXTRACTION REPORT")
print("=" * 60)

print(
    f"Original records retained: "
    f"{len(original_df)}"
)

print(
    f"Repeated prompt texts found: "
    f"{repeated_prompt_count}"
)

print(
    f"Repeated complete records found: "
    f"{repeated_full_record_count}"
)

print()
print("Important: repeated records were NOT removed.")
print()

print(f"CSV saved:   {OUTPUT_CSV}")
print(f"JSONL saved: {OUTPUT_JSONL}")

print()
display(original_df.head(10))


# =====================================
# Download
# =====================================

files.download(OUTPUT_CSV)
files.download(OUTPUT_JSONL)

MMLU ORIGINAL EXTRACTION REPORT
Original records retained: 14042
Repeated prompt texts found: 174
Repeated complete records found: 27

Important: repeated records were NOT removed.

CSV saved:   /content/mmlu_original_reference_full.csv
JSONL saved: /content/mmlu_original_reference_full.jsonl



,Original_ID,Original_Row,Original_Subject,Original_Prompt,Original_Choices,Gold_Answer
0,MMLU_00001,1,abstract_algebra,Find the degree for the given field extension ...,"[""0"", ""4"", ""2"", ""6""]",1
1,MMLU_00002,2,abstract_algebra,"Let p = (1, 2, 5, 4)(2, 3) in S_5 . Find the i...","[""8"", ""2"", ""24"", ""120""]",2
2,MMLU_00003,3,abstract_algebra,Find all zeros in the indicated finite field o...,"[""0"", ""1"", ""0,1"", ""0,4""]",3
3,MMLU_00004,4,abstract_algebra,Statement 1 | A factor group of a non-Abelian ...,"[""True, True"", ""False, False"", ""True, False"", ...",1
4,MMLU_00005,5,abstract_algebra,Find the product of the given polynomials in t...,"[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1
5,MMLU_00006,6,abstract_algebra,Statement 1 | If a group has an element of ord...,"[""True, True"", ""False, False"", ""True, False"", ...",0
6,MMLU_00007,7,abstract_algebra,Statement 1 | Every homomorphic image of a gro...,"[""True, True"", ""False, False"", ""True, False"", ...",0
7,MMLU_00008,8,abstract_algebra,Statement 1 | A ring homomorphism is one to on...,"[""True, True"", ""False, False"", ""True, False"", ...",3
8,MMLU_00009,9,abstract_algebra,Find the degree for the given field extension ...,"[""0"", ""4"", ""2"", ""6""]",1
9,MMLU_00010,10,abstract_algebra,Find all zeros in the indicated finite field o...,"[""1"", ""2"", ""2,3"", ""6""]",2


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [17]:
# ============================================================
# Install dependency
# ============================================================

!pip -q install rapidfuzz


# ============================================================
# Imports
# ============================================================

import os
import re
import json
import shutil
import unicodedata
from collections import defaultdict

import pandas as pd
from rapidfuzz import fuzz, process
from google.colab import files


# ============================================================
# Settings
# ============================================================

ORIGINAL_FILE = (
    "/content/mmlu_original_reference_full.jsonl"
)

LANGUAGE_FILES = {
    "Arabic": "/content/arabic_A.jsonl",
    "Chinese": "/content/chinese_A.jsonl",
    "French": "/content/french_A.jsonl",
    "German": "/content/german_A.jsonl",
    "Japanese": "/content/japanese_A.jsonl",
    "Portuguese": "/content/portuguese_A.jsonl",
    "Russian": "/content/russian_A.jsonl",
    "Spanish": "/content/spanish_A.jsonl",
}

OUTPUT_DIR = "/content/mmlu_match_output"

COMBINED_CSV = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_match_audit_all.csv"
)

COMBINED_JSONL = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_match_audit_all.jsonl"
)

LOW_CONFIDENCE_CSV = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_low_confidence.csv"
)

METADATA_PROBLEM_CSV = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_metadata_problems.csv"
)

SUMMARY_CSV = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_match_summary.csv"
)

ZIP_BASE = "/content/mmlu_match_output"


# ============================================================
# Matching thresholds
# ============================================================

# High confidence:
# strong text similarity and clear separation from second match
HIGH_SCORE = 75
HIGH_MARGIN = 5

# Scores below this are treated as low confidence
REVIEW_SCORE = 55


# ============================================================
# Helper functions
# ============================================================

def load_jsonl(file_path):
    """
    Read a JSONL file safely.
    Also preserve the original line number.
    """

    records = []
    errors = []

    with open(
        file_path,
        "r",
        encoding="utf-8-sig"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:
                errors.append(
                    {
                        "Line": line_number,
                        "Error": str(error),
                    }
                )
                continue

            record["_Source_Row"] = line_number
            records.append(record)

    return records, errors


def normalize_text(value):
    """
    Normalize text for fuzzy matching.
    Does not remove mathematical symbols.
    """

    if value is None:
        return ""

    text = str(value)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.strip()

    # Remove surrounding quotation marks
    while (
        len(text) >= 2
        and text[0] in ['"', "'"]
        and text[-1] == text[0]
    ):
        text = text[1:-1].strip()

    text = text.lower()

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def parse_choices(value):
    """
    Convert choices into a Python list.
    Handles both actual lists and JSON strings.
    """

    if isinstance(value, list):
        return value

    if value is None:
        return []

    if isinstance(value, str):

        value = value.strip()

        try:
            parsed = json.loads(value)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return [value]


def choices_signature(value):
    """
    Create a normalized signature for answer choices.
    """

    choices = parse_choices(value)

    normalized_choices = [
        normalize_text(choice)
        for choice in choices
    ]

    return " || ".join(normalized_choices)


def normalize_answer(value):
    """
    Normalize answer index.
    """

    if value is None:
        return ""

    try:
        return str(int(value))

    except Exception:
        return normalize_text(value)


def yes_no(value):
    """
    Convert Boolean value into Yes or No.
    """

    return "Yes" if bool(value) else "No"


# ============================================================
# Prepare output directory
# ============================================================

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# Load original reference
# ============================================================

original_records, original_errors = load_jsonl(
    ORIGINAL_FILE
)

if original_errors:
    print(
        f"Warning: {len(original_errors)} "
        f"JSON errors in original file."
    )

original_df = pd.DataFrame(
    original_records
)


# Required columns
required_original_columns = [
    "Original_ID",
    "Original_Row",
    "Original_Subject",
    "Original_Prompt",
    "Original_Choices",
    "Gold_Answer",
]

missing_original_columns = [
    column
    for column in required_original_columns
    if column not in original_df.columns
]

if missing_original_columns:
    raise ValueError(
        "Missing columns in original reference: "
        + ", ".join(missing_original_columns)
    )


# Normalize original fields
original_df["_Prompt_Normalized"] = (
    original_df["Original_Prompt"]
    .apply(normalize_text)
)

original_df["_Subject_Normalized"] = (
    original_df["Original_Subject"]
    .apply(normalize_text)
)

original_df["_Choices_Signature"] = (
    original_df["Original_Choices"]
    .apply(choices_signature)
)

original_df["_Answer_Normalized"] = (
    original_df["Gold_Answer"]
    .apply(normalize_answer)
)


print("=" * 70)
print("ORIGINAL REFERENCE")
print("=" * 70)
print(f"Original records loaded: {len(original_df)}")
print()


# ============================================================
# Create original candidate indexes
# ============================================================

indices_by_subject = defaultdict(list)
indices_by_choices = defaultdict(list)
indices_by_choices_answer = defaultdict(list)
indices_by_answer = defaultdict(list)

for original_index, row in original_df.iterrows():

    subject_key = row["_Subject_Normalized"]
    choices_key = row["_Choices_Signature"]
    answer_key = row["_Answer_Normalized"]

    indices_by_subject[
        subject_key
    ].append(original_index)

    indices_by_choices[
        choices_key
    ].append(original_index)

    indices_by_choices_answer[
        (choices_key, answer_key)
    ].append(original_index)

    indices_by_answer[
        answer_key
    ].append(original_index)


# ============================================================
# Match one ESL record
# ============================================================

def match_one_esl_record(
    esl_record,
    language
):
    """
    Match a single ESL question to an original MMLU record.

    Candidate pool includes:
    1. Original records from the same subject
    2. Records with the same choices
    3. Records with the same choices and answer

    Question similarity remains the main matching signal.
    """

    esl_row = esl_record.get(
        "_Source_Row",
        None
    )

    esl_question = esl_record.get(
        "question",
        ""
    )

    esl_subject = esl_record.get(
        "subject",
        ""
    )

    esl_choices = esl_record.get(
        "choices",
        []
    )

    esl_answer = esl_record.get(
        "answer",
        None
    )

    question_normalized = normalize_text(
        esl_question
    )

    subject_normalized = normalize_text(
        esl_subject
    )

    choices_normalized = choices_signature(
        esl_choices
    )

    answer_normalized = normalize_answer(
        esl_answer
    )

    # --------------------------------------------------------
    # Empty question
    # --------------------------------------------------------

    if not question_normalized:

        return {
            "Language": language,
            "ESL_Row": esl_row,
            "ESL_Subject": esl_subject,
            "ESL_Question": esl_question,
            "ESL_Choices": json.dumps(
                parse_choices(esl_choices),
                ensure_ascii=False
            ),
            "ESL_Answer": esl_answer,

            "Matched_Original_ID": "",
            "Original_Row": "",
            "Original_Subject": "",
            "Original_Prompt": "",
            "Original_Choices": "",
            "Gold_Answer": "",

            "Question_Similarity": 0,
            "Second_Best_Similarity": 0,
            "Similarity_Margin": 0,

            "Subject_Preserved": "No",
            "Choices_Preserved": "No",
            "Answer_Preserved": "No",

            "Match_Status": "LOW",
            "Integrity_Status": "EMPTY_QUESTION",
            "Candidate_Count": 0,
        }

    # --------------------------------------------------------
    # Build candidate pool
    # --------------------------------------------------------

    candidate_indices = set()

    # Same subject
    candidate_indices.update(
        indices_by_subject.get(
            subject_normalized,
            []
        )
    )

    # Same choices, regardless of subject
    candidate_indices.update(
        indices_by_choices.get(
            choices_normalized,
            []
        )
    )

    # Same choices and answer
    candidate_indices.update(
        indices_by_choices_answer.get(
            (
                choices_normalized,
                answer_normalized
            ),
            []
        )
    )

    # Fallback: same answer
    if not candidate_indices:
        candidate_indices.update(
            indices_by_answer.get(
                answer_normalized,
                []
            )
        )

    candidate_indices = sorted(
        candidate_indices
    )

    if not candidate_indices:

        return {
            "Language": language,
            "ESL_Row": esl_row,
            "ESL_Subject": esl_subject,
            "ESL_Question": esl_question,
            "ESL_Choices": json.dumps(
                parse_choices(esl_choices),
                ensure_ascii=False
            ),
            "ESL_Answer": esl_answer,

            "Matched_Original_ID": "",
            "Original_Row": "",
            "Original_Subject": "",
            "Original_Prompt": "",
            "Original_Choices": "",
            "Gold_Answer": "",

            "Question_Similarity": 0,
            "Second_Best_Similarity": 0,
            "Similarity_Margin": 0,

            "Subject_Preserved": "No",
            "Choices_Preserved": "No",
            "Answer_Preserved": "No",

            "Match_Status": "LOW",
            "Integrity_Status": "NO_CANDIDATES",
            "Candidate_Count": 0,
        }

    # --------------------------------------------------------
    # First retrieve top candidates using question similarity
    # --------------------------------------------------------

    candidate_prompts = [
        original_df.at[
            original_index,
            "_Prompt_Normalized"
        ]
        for original_index in candidate_indices
    ]

    # Retrieve top 10 text candidates
    fuzzy_matches = process.extract(
        question_normalized,
        candidate_prompts,
        scorer=fuzz.WRatio,
        limit=min(
            10,
            len(candidate_prompts)
        )
    )

    ranked_candidates = []

    for _, text_score, local_index in fuzzy_matches:

        original_index = candidate_indices[
            local_index
        ]

        original_row = original_df.loc[
            original_index
        ]

        subject_match = (
            subject_normalized
            == original_row[
                "_Subject_Normalized"
            ]
        )

        choices_match = (
            choices_normalized
            == original_row[
                "_Choices_Signature"
            ]
        )

        answer_match = (
            answer_normalized
            == original_row[
                "_Answer_Normalized"
            ]
        )

        # Text similarity is the primary signal.
        # Metadata adds only a small bonus.
        ranking_score = float(text_score)

        if choices_match:
            ranking_score += 4

        if answer_match:
            ranking_score += 1.5

        if subject_match:
            ranking_score += 1

        ranked_candidates.append(
            {
                "Original_Index": original_index,
                "Text_Score": float(text_score),
                "Ranking_Score": ranking_score,
                "Subject_Match": subject_match,
                "Choices_Match": choices_match,
                "Answer_Match": answer_match,
            }
        )

    ranked_candidates = sorted(
        ranked_candidates,
        key=lambda item: item[
            "Ranking_Score"
        ],
        reverse=True
    )

    best = ranked_candidates[0]

    if len(ranked_candidates) >= 2:
        second = ranked_candidates[1]
        second_text_score = second[
            "Text_Score"
        ]
        ranking_margin = (
            best["Ranking_Score"]
            - second["Ranking_Score"]
        )

    else:
        second_text_score = 0
        ranking_margin = best[
            "Ranking_Score"
        ]

    matched_original = original_df.loc[
        best["Original_Index"]
    ]

    # --------------------------------------------------------
    # Confidence status
    # --------------------------------------------------------

    if (
        best["Text_Score"] >= HIGH_SCORE
        and ranking_margin >= HIGH_MARGIN
    ):
        match_status = "HIGH"

    elif best["Text_Score"] >= REVIEW_SCORE:
        match_status = "REVIEW"

    else:
        match_status = "LOW"

    # --------------------------------------------------------
    # Integrity status
    # --------------------------------------------------------

    problems = []

    if not best["Subject_Match"]:
        problems.append(
            "SUBJECT_MISMATCH"
        )

    if not best["Choices_Match"]:
        problems.append(
            "CHOICES_MISMATCH"
        )

    if not best["Answer_Match"]:
        problems.append(
            "ANSWER_MISMATCH"
        )

    if problems:
        integrity_status = ";".join(
            problems
        )

    else:
        integrity_status = "OK"

    return {
        "Language": language,
        "ESL_Row": esl_row,
        "ESL_Subject": esl_subject,
        "ESL_Question": esl_question,
        "ESL_Choices": json.dumps(
            parse_choices(esl_choices),
            ensure_ascii=False
        ),
        "ESL_Answer": esl_answer,

        "Matched_Original_ID": (
            matched_original[
                "Original_ID"
            ]
        ),
        "Original_Row": (
            matched_original[
                "Original_Row"
            ]
        ),
        "Original_Subject": (
            matched_original[
                "Original_Subject"
            ]
        ),
        "Original_Prompt": (
            matched_original[
                "Original_Prompt"
            ]
        ),
        "Original_Choices": (
            matched_original[
                "Original_Choices"
            ]
        ),
        "Gold_Answer": (
            matched_original[
                "Gold_Answer"
            ]
        ),

        "Question_Similarity": round(
            best["Text_Score"],
            2
        ),
        "Second_Best_Similarity": round(
            second_text_score,
            2
        ),
        "Similarity_Margin": round(
            ranking_margin,
            2
        ),

        "Subject_Preserved": yes_no(
            best["Subject_Match"]
        ),
        "Choices_Preserved": yes_no(
            best["Choices_Match"]
        ),
        "Answer_Preserved": yes_no(
            best["Answer_Match"]
        ),

        "Match_Status": match_status,
        "Integrity_Status": integrity_status,
        "Candidate_Count": len(
            candidate_indices
        ),
    }


# ============================================================
# Process all eight languages
# ============================================================

all_language_results = []
summary_rows = []

for language, file_path in LANGUAGE_FILES.items():

    print("=" * 70)
    print(f"Processing: {language}")
    print(f"File: {file_path}")
    print("=" * 70)

    if not os.path.exists(file_path):

        print(
            f"Warning: file not found: {file_path}"
        )

        summary_rows.append(
            {
                "Language": language,
                "Input_Rows": 0,
                "High_Confidence": 0,
                "Needs_Review": 0,
                "Low_Confidence": 0,
                "Integrity_OK": 0,
                "Metadata_Problems": 0,
                "Duplicate_Assignments": 0,
                "JSON_Errors": "",
                "File_Status": "FILE_NOT_FOUND",
            }
        )

        continue

    esl_records, esl_errors = load_jsonl(
        file_path
    )

    language_results = []

    for count, esl_record in enumerate(
        esl_records,
        start=1
    ):

        matched_record = match_one_esl_record(
            esl_record,
            language
        )

        language_results.append(
            matched_record
        )

        if count % 500 == 0:
            print(
                f"Matched {count} records..."
            )

    language_df = pd.DataFrame(
        language_results
    )

    # Detect duplicate assignment of one original record
    # within the same language file.
    language_df[
        "Duplicate_Original_Assignment"
    ] = "No"

    valid_match_mask = (
        language_df[
            "Matched_Original_ID"
        ].astype(str).str.len() > 0
    )

    duplicate_mask = (
        valid_match_mask
        & language_df.duplicated(
            subset=[
                "Matched_Original_ID"
            ],
            keep=False
        )
    )

    language_df.loc[
        duplicate_mask,
        "Duplicate_Original_Assignment"
    ] = "Yes"

    # Duplicate assignment should be reviewed,
    # even when text similarity is high.
    language_df.loc[
        duplicate_mask
        & (
            language_df[
                "Match_Status"
            ] == "HIGH"
        ),
        "Match_Status"
    ] = "REVIEW"

    # Save one audit file per language
    language_output_file = os.path.join(
        OUTPUT_DIR,
        (
            f"mmlu_{language.lower()}_"
            f"match_audit.csv"
        )
    )

    language_df.to_csv(
        language_output_file,
        index=False,
        encoding="utf-8-sig"
    )

    all_language_results.append(
        language_df
    )

    summary_rows.append(
        {
            "Language": language,
            "Input_Rows": len(
                language_df
            ),
            "High_Confidence": int(
                (
                    language_df[
                        "Match_Status"
                    ] == "HIGH"
                ).sum()
            ),
            "Needs_Review": int(
                (
                    language_df[
                        "Match_Status"
                    ] == "REVIEW"
                ).sum()
            ),
            "Low_Confidence": int(
                (
                    language_df[
                        "Match_Status"
                    ] == "LOW"
                ).sum()
            ),
            "Integrity_OK": int(
                (
                    language_df[
                        "Integrity_Status"
                    ] == "OK"
                ).sum()
            ),
            "Metadata_Problems": int(
                (
                    language_df[
                        "Integrity_Status"
                    ] != "OK"
                ).sum()
            ),
            "Duplicate_Assignments": int(
                duplicate_mask.sum()
            ),
            "JSON_Errors": len(
                esl_errors
            ),
            "File_Status": "OK",
        }
    )

    print(
        f"Rows: {len(language_df)}"
    )

    print(
        "High confidence: "
        f"{(language_df['Match_Status'] == 'HIGH').sum()}"
    )

    print(
        "Needs review: "
        f"{(language_df['Match_Status'] == 'REVIEW').sum()}"
    )

    print(
        "Low confidence: "
        f"{(language_df['Match_Status'] == 'LOW').sum()}"
    )

    print(
        "Metadata problems: "
        f"{(language_df['Integrity_Status'] != 'OK').sum()}"
    )

    print()


# ============================================================
# Combine all languages
# ============================================================

if not all_language_results:
    raise ValueError(
        "No language files were processed."
    )

combined_df = pd.concat(
    all_language_results,
    ignore_index=True
)

summary_df = pd.DataFrame(
    summary_rows
)


# ============================================================
# Create review files
# ============================================================

low_confidence_df = combined_df[
    combined_df[
        "Match_Status"
    ].isin(
        [
            "REVIEW",
            "LOW",
        ]
    )
].copy()

metadata_problem_df = combined_df[
    (
        combined_df[
            "Integrity_Status"
        ] != "OK"
    )
    |
    (
        combined_df[
            "Duplicate_Original_Assignment"
        ] == "Yes"
    )
].copy()


# ============================================================
# Save combined outputs
# ============================================================

combined_df.to_csv(
    COMBINED_CSV,
    index=False,
    encoding="utf-8-sig"
)

combined_df.to_json(
    COMBINED_JSONL,
    orient="records",
    lines=True,
    force_ascii=False
)

low_confidence_df.to_csv(
    LOW_CONFIDENCE_CSV,
    index=False,
    encoding="utf-8-sig"
)

metadata_problem_df.to_csv(
    METADATA_PROBLEM_CSV,
    index=False,
    encoding="utf-8-sig"
)

summary_df.to_csv(
    SUMMARY_CSV,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# Final report
# ============================================================

print("=" * 70)
print("MMLU ESL MATCHING SUMMARY")
print("=" * 70)

display(summary_df)

print()
print(f"Total ESL rows: {len(combined_df)}")

print(
    "High-confidence matches: "
    f"{(combined_df['Match_Status'] == 'HIGH').sum()}"
)

print(
    "Review matches: "
    f"{(combined_df['Match_Status'] == 'REVIEW').sum()}"
)

print(
    "Low-confidence matches: "
    f"{(combined_df['Match_Status'] == 'LOW').sum()}"
)

print(
    "Metadata problems: "
    f"{len(metadata_problem_df)}"
)

print()
print(f"Combined CSV: {COMBINED_CSV}")
print(f"Summary CSV: {SUMMARY_CSV}")
print(f"Low-confidence CSV: {LOW_CONFIDENCE_CSV}")
print(f"Metadata-problem CSV: {METADATA_PROBLEM_CSV}")


# ============================================================
# Zip and download all outputs
# ============================================================

zip_file = shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

print()
print(f"ZIP saved: {zip_file}")

files.download(
    zip_file
)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 36.2 MB/s eta 0:00:00
ORIGINAL REFERENCE
Original records loaded: 14042

Processing: Arabic
File: /content/arabic_A.jsonl
Matched 500 records...
Matched 1000 records...
Matched 1500 records...
Matched 2000 records...
Matched 2500 records...
Matched 3000 records...
Matched 3500 records...
Matched 4000 records...
Matched 4500 records...
Matched 5000 records...


KeyboardInterrupt: 

In [18]:
# ============================================================
# MMLU FAST ESL–ORIGINAL MATCHING
# ============================================================

import os
import re
import json
import shutil
import unicodedata
from collections import defaultdict

import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from google.colab import files


# ============================================================
# 1. Settings
# ============================================================

ORIGINAL_FILE = (
    "/content/mmlu_original_reference_full.jsonl"
)

LANGUAGE_FILES = {
    "Arabic": "/content/arabic_A.jsonl",
    "Chinese": "/content/chinese_A.jsonl",
    "French": "/content/french_A.jsonl",
    "German": "/content/german_A.jsonl",
    "Japanese": "/content/japanese_A.jsonl",
    "Portuguese": "/content/portuguese_A.jsonl",
    "Russian": "/content/russian_A.jsonl",
    "Spanish": "/content/spanish_A.jsonl",
}

OUTPUT_DIR = "/content/mmlu_fast_match_output"
ZIP_BASE = "/content/mmlu_fast_match_output"

# Number of nearest original candidates retained per ESL item
TOP_K = 20

# Confidence thresholds
HIGH_THRESHOLD = 0.70
REVIEW_THRESHOLD = 0.45

# A subject-level result below this score is sent to global fallback
GLOBAL_FALLBACK_THRESHOLD = 0.32


# ============================================================
# 2. Helper functions
# ============================================================

def load_jsonl(file_path):
    """
    Safely load a JSONL file and preserve source row numbers.
    """

    records = []
    errors = []

    with open(
        file_path,
        "r",
        encoding="utf-8-sig"
    ) as file:

        for line_number, line in enumerate(
            file,
            start=1
        ):

            line = line.strip()

            if not line:
                continue

            try:
                record = json.loads(line)

            except json.JSONDecodeError as error:

                errors.append(
                    {
                        "Line": line_number,
                        "Error": str(error),
                    }
                )

                continue

            record["_Source_Row"] = line_number
            records.append(record)

    return records, errors


def normalize_text(value):
    """
    Normalize text while preserving mathematical symbols.
    """

    if value is None:
        return ""

    text = str(value)

    text = unicodedata.normalize(
        "NFKC",
        text
    )

    text = text.strip()

    # Remove repeated surrounding quotation marks
    while (
        len(text) >= 2
        and text[0] in ['"', "'"]
        and text[-1] == text[0]
    ):
        text = text[1:-1].strip()

    text = text.lower()

    # Normalize some punctuation
    text = text.replace("–", "-")
    text = text.replace("—", "-")
    text = text.replace("−", "-")

    text = re.sub(
        r"\s+",
        " ",
        text
    )

    return text.strip()


def parse_choices(value):
    """
    Convert choices to a Python list.
    """

    if isinstance(value, list):
        return value

    if value is None:
        return []

    if isinstance(value, str):

        value = value.strip()

        try:
            parsed = json.loads(value)

            if isinstance(parsed, list):
                return parsed

        except Exception:
            pass

    return [value]


def choices_signature(value):
    """
    Create a normalized choices signature.
    """

    choices = parse_choices(value)

    normalized = [
        normalize_text(choice)
        for choice in choices
    ]

    return " || ".join(normalized)


def normalize_answer(value):
    """
    Normalize the answer index.
    """

    if value is None:
        return ""

    try:
        return str(int(value))

    except Exception:
        return normalize_text(value)


def yes_no(value):
    return "Yes" if bool(value) else "No"


def safe_similarity(value):
    """
    Ensure similarity remains between 0 and 1.
    """

    return float(
        max(
            0.0,
            min(1.0, value)
        )
    )


# ============================================================
# 3. Prepare output directory
# ============================================================

if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)

os.makedirs(
    OUTPUT_DIR,
    exist_ok=True
)


# ============================================================
# 4. Load original reference
# ============================================================

original_records, original_errors = load_jsonl(
    ORIGINAL_FILE
)

if original_errors:

    print(
        f"Warning: {len(original_errors)} "
        "JSON errors in the original file."
    )


original_df = pd.DataFrame(
    original_records
)


required_columns = [
    "Original_ID",
    "Original_Row",
    "Original_Subject",
    "Original_Prompt",
    "Original_Choices",
    "Gold_Answer",
]


missing_columns = [
    column
    for column in required_columns
    if column not in original_df.columns
]


if missing_columns:

    raise ValueError(
        "Missing original columns: "
        + ", ".join(missing_columns)
    )


original_df["_Prompt_Normalized"] = (
    original_df["Original_Prompt"]
    .apply(normalize_text)
)

original_df["_Subject_Normalized"] = (
    original_df["Original_Subject"]
    .apply(normalize_text)
)

original_df["_Choices_Signature"] = (
    original_df["Original_Choices"]
    .apply(choices_signature)
)

original_df["_Answer_Normalized"] = (
    original_df["Gold_Answer"]
    .apply(normalize_answer)
)


print("=" * 70)
print("ORIGINAL REFERENCE")
print("=" * 70)

print(
    f"Original records loaded: "
    f"{len(original_df)}"
)

print(
    f"Original subjects: "
    f"{original_df['_Subject_Normalized'].nunique()}"
)

print()


# ============================================================
# 5. Build global fallback model once
# ============================================================

print("=" * 70)
print("BUILDING GLOBAL FALLBACK INDEX")
print("=" * 70)

print(
    "This is built once and is only used for "
    "difficult or incorrectly labelled rows."
)


global_vectorizer = TfidfVectorizer(
    analyzer="char_wb",
    ngram_range=(3, 5),
    min_df=1,
    max_features=120000,
    sublinear_tf=True,
    dtype=np.float32,
)

global_original_matrix = (
    global_vectorizer.fit_transform(
        original_df["_Prompt_Normalized"]
    )
)


global_neighbors = NearestNeighbors(
    n_neighbors=min(
        TOP_K,
        len(original_df)
    ),
    metric="cosine",
    algorithm="brute",
    n_jobs=-1,
)

global_neighbors.fit(
    global_original_matrix
)

print(
    "Global fallback index ready."
)

print()


# ============================================================
# 6. Match one subject group
# ============================================================

def subject_level_candidates(
    original_group,
    esl_group
):
    """
    Match one subject using batch TF-IDF nearest-neighbour search.

    Returns candidate edges rather than directly assigning rows.
    """

    original_texts = (
        original_group[
            "_Prompt_Normalized"
        ].tolist()
    )

    esl_texts = (
        esl_group[
            "_Question_Normalized"
        ].tolist()
    )

    if not original_texts or not esl_texts:
        return []

    combined_texts = (
        original_texts
        + esl_texts
    )

    vectorizer = TfidfVectorizer(
        analyzer="char_wb",
        ngram_range=(3, 5),
        min_df=1,
        sublinear_tf=True,
        dtype=np.float32,
    )

    try:

        combined_matrix = (
            vectorizer.fit_transform(
                combined_texts
            )
        )

    except ValueError:

        return []

    original_count = len(original_texts)

    original_matrix = (
        combined_matrix[
            :original_count
        ]
    )

    esl_matrix = (
        combined_matrix[
            original_count:
        ]
    )

    neighbour_count = min(
        TOP_K,
        original_count
    )

    model = NearestNeighbors(
        n_neighbors=neighbour_count,
        metric="cosine",
        algorithm="brute",
        n_jobs=-1,
    )

    model.fit(
        original_matrix
    )

    distances, neighbours = model.kneighbors(
        esl_matrix
    )

    original_indices = (
        original_group.index.tolist()
    )

    esl_indices = (
        esl_group.index.tolist()
    )

    edges = []

    original_group_size = max(
        len(original_group) - 1,
        1
    )

    esl_group_size = max(
        len(esl_group) - 1,
        1
    )

    for esl_local_index, esl_global_index in enumerate(
        esl_indices
    ):

        esl_choices = esl_group.at[
            esl_global_index,
            "_Choices_Signature"
        ]

        esl_answer = esl_group.at[
            esl_global_index,
            "_Answer_Normalized"
        ]

        esl_relative_position = (
            esl_local_index
            / esl_group_size
        )

        for rank in range(
            neighbour_count
        ):

            original_local_index = int(
                neighbours[
                    esl_local_index,
                    rank
                ]
            )

            original_global_index = (
                original_indices[
                    original_local_index
                ]
            )

            similarity = safe_similarity(
                1.0
                - distances[
                    esl_local_index,
                    rank
                ]
            )

            choices_match = (
                esl_choices
                == original_df.at[
                    original_global_index,
                    "_Choices_Signature"
                ]
            )

            answer_match = (
                esl_answer
                == original_df.at[
                    original_global_index,
                    "_Answer_Normalized"
                ]
            )

            original_relative_position = (
                original_local_index
                / original_group_size
            )

            position_difference = abs(
                esl_relative_position
                - original_relative_position
            )

            position_bonus = (
                0.015
                * (
                    1.0
                    - min(
                        position_difference,
                        1.0
                    )
                )
            )

            # Question similarity remains the main signal.
            # Choices and answers only help resolve close ties.
            adjusted_score = similarity

            if choices_match:
                adjusted_score += 0.025

            if answer_match:
                adjusted_score += 0.010

            adjusted_score += position_bonus

            edges.append(
                {
                    "ESL_Index": esl_global_index,
                    "Original_Index": original_global_index,
                    "Similarity": similarity,
                    "Adjusted_Score": adjusted_score,
                    "Candidate_Rank": rank + 1,
                }
            )

    return edges


# ============================================================
# 7. Greedy one-to-one assignment
# ============================================================

def assign_candidate_edges(
    edges,
    blocked_original_indices=None
):
    """
    Assign each ESL row to at most one original row.
    Each original row is also used at most once.
    """

    if blocked_original_indices is None:
        blocked_original_indices = set()

    assignments = {}

    assigned_esl = set()
    assigned_original = set(
        blocked_original_indices
    )

    sorted_edges = sorted(
        edges,
        key=lambda item: (
            item["Adjusted_Score"],
            item["Similarity"]
        ),
        reverse=True
    )

    for edge in sorted_edges:

        esl_index = edge["ESL_Index"]
        original_index = edge["Original_Index"]

        if esl_index in assigned_esl:
            continue

        if original_index in assigned_original:
            continue

        assignments[
            esl_index
        ] = edge

        assigned_esl.add(
            esl_index
        )

        assigned_original.add(
            original_index
        )

    return assignments


# ============================================================
# 8. Global fallback for difficult rows
# ============================================================

def global_fallback_matches(
    language_df,
    unresolved_indices,
    already_used_original_indices
):
    """
    Search difficult rows globally across all original prompts.
    """

    fallback_assignments = {}

    if not unresolved_indices:
        return fallback_assignments

    unresolved_texts = (
        language_df.loc[
            unresolved_indices,
            "_Question_Normalized"
        ]
        .tolist()
    )

    unresolved_matrix = (
        global_vectorizer.transform(
            unresolved_texts
        )
    )

    distances, neighbours = (
        global_neighbors.kneighbors(
            unresolved_matrix
        )
    )

    candidate_edges = []

    for local_index, esl_index in enumerate(
        unresolved_indices
    ):

        esl_choices = language_df.at[
            esl_index,
            "_Choices_Signature"
        ]

        esl_answer = language_df.at[
            esl_index,
            "_Answer_Normalized"
        ]

        esl_subject = language_df.at[
            esl_index,
            "_Subject_Normalized"
        ]

        for rank in range(
            neighbours.shape[1]
        ):

            original_index = int(
                neighbours[
                    local_index,
                    rank
                ]
            )

            if (
                original_index
                in already_used_original_indices
            ):
                continue

            similarity = safe_similarity(
                1.0
                - distances[
                    local_index,
                    rank
                ]
            )

            choices_match = (
                esl_choices
                == original_df.at[
                    original_index,
                    "_Choices_Signature"
                ]
            )

            answer_match = (
                esl_answer
                == original_df.at[
                    original_index,
                    "_Answer_Normalized"
                ]
            )

            subject_match = (
                esl_subject
                == original_df.at[
                    original_index,
                    "_Subject_Normalized"
                ]
            )

            adjusted_score = similarity

            if choices_match:
                adjusted_score += 0.025

            if answer_match:
                adjusted_score += 0.010

            if subject_match:
                adjusted_score += 0.005

            candidate_edges.append(
                {
                    "ESL_Index": esl_index,
                    "Original_Index": original_index,
                    "Similarity": similarity,
                    "Adjusted_Score": adjusted_score,
                    "Candidate_Rank": rank + 1,
                }
            )

    fallback_assignments = assign_candidate_edges(
        candidate_edges,
        blocked_original_indices=(
            already_used_original_indices
        )
    )

    return fallback_assignments


# ============================================================
# 9. Create one output row
# ============================================================

def build_output_row(
    language,
    language_df,
    esl_index,
    assignment,
    second_best_similarity
):
    """
    Combine ESL and matched original information.
    """

    esl_row = language_df.loc[
        esl_index
    ]

    if assignment is None:

        return {
            "Language": language,
            "ESL_Row": esl_row["_Source_Row"],
            "ESL_Subject": esl_row.get(
                "subject",
                ""
            ),
            "ESL_Question": esl_row.get(
                "question",
                ""
            ),
            "ESL_Choices": json.dumps(
                parse_choices(
                    esl_row.get(
                        "choices",
                        []
                    )
                ),
                ensure_ascii=False
            ),
            "ESL_Answer": esl_row.get(
                "answer",
                None
            ),

            "Matched_Original_ID": "",
            "Original_Row": "",
            "Original_Subject": "",
            "Original_Prompt": "",
            "Original_Choices": "",
            "Gold_Answer": "",

            "Question_Similarity": 0.0,
            "Second_Best_Similarity": round(
                second_best_similarity,
                4
            ),
            "Similarity_Margin": 0.0,
            "Candidate_Rank": "",

            "Subject_Preserved": "No",
            "Choices_Preserved": "No",
            "Answer_Preserved": "No",

            "Match_Status": "LOW",
            "Integrity_Status": "UNMATCHED",
        }

    original_index = assignment[
        "Original_Index"
    ]

    original_row = original_df.loc[
        original_index
    ]

    similarity = assignment[
        "Similarity"
    ]

    margin = (
        similarity
        - second_best_similarity
    )

    subject_match = (
        esl_row[
            "_Subject_Normalized"
        ]
        == original_row[
            "_Subject_Normalized"
        ]
    )

    choices_match = (
        esl_row[
            "_Choices_Signature"
        ]
        == original_row[
            "_Choices_Signature"
        ]
    )

    answer_match = (
        esl_row[
            "_Answer_Normalized"
        ]
        == original_row[
            "_Answer_Normalized"
        ]
    )

    if similarity >= HIGH_THRESHOLD:
        match_status = "HIGH"

    elif similarity >= REVIEW_THRESHOLD:
        match_status = "REVIEW"

    else:
        match_status = "LOW"

    integrity_problems = []

    if not subject_match:
        integrity_problems.append(
            "SUBJECT_MISMATCH"
        )

    if not choices_match:
        integrity_problems.append(
            "CHOICES_MISMATCH"
        )

    if not answer_match:
        integrity_problems.append(
            "ANSWER_MISMATCH"
        )

    if integrity_problems:

        integrity_status = ";".join(
            integrity_problems
        )

    else:
        integrity_status = "OK"

    return {
        "Language": language,
        "ESL_Row": esl_row["_Source_Row"],
        "ESL_Subject": esl_row.get(
            "subject",
            ""
        ),
        "ESL_Question": esl_row.get(
            "question",
            ""
        ),
        "ESL_Choices": json.dumps(
            parse_choices(
                esl_row.get(
                    "choices",
                    []
                )
            ),
            ensure_ascii=False
        ),
        "ESL_Answer": esl_row.get(
            "answer",
            None
        ),

        "Matched_Original_ID": (
            original_row[
                "Original_ID"
            ]
        ),
        "Original_Row": (
            original_row[
                "Original_Row"
            ]
        ),
        "Original_Subject": (
            original_row[
                "Original_Subject"
            ]
        ),
        "Original_Prompt": (
            original_row[
                "Original_Prompt"
            ]
        ),
        "Original_Choices": (
            original_row[
                "Original_Choices"
            ]
        ),
        "Gold_Answer": (
            original_row[
                "Gold_Answer"
            ]
        ),

        "Question_Similarity": round(
            similarity,
            4
        ),
        "Second_Best_Similarity": round(
            second_best_similarity,
            4
        ),
        "Similarity_Margin": round(
            margin,
            4
        ),
        "Candidate_Rank": assignment[
            "Candidate_Rank"
        ],

        "Subject_Preserved": yes_no(
            subject_match
        ),
        "Choices_Preserved": yes_no(
            choices_match
        ),
        "Answer_Preserved": yes_no(
            answer_match
        ),

        "Match_Status": match_status,
        "Integrity_Status": integrity_status,
    }


# ============================================================
# 10. Process all language files
# ============================================================

all_language_outputs = []
summary_rows = []


for language, file_path in LANGUAGE_FILES.items():

    print("=" * 70)
    print(f"PROCESSING: {language}")
    print("=" * 70)

    if not os.path.exists(file_path):

        print(
            f"File not found: {file_path}"
        )

        summary_rows.append(
            {
                "Language": language,
                "Input_Rows": 0,
                "Matched_Rows": 0,
                "High_Confidence": 0,
                "Needs_Review": 0,
                "Low_Confidence": 0,
                "Integrity_OK": 0,
                "Metadata_Problems": 0,
                "Unmatched": 0,
                "JSON_Errors": "",
                "File_Status": "FILE_NOT_FOUND",
            }
        )

        continue

    esl_records, esl_errors = load_jsonl(
        file_path
    )

    language_df = pd.DataFrame(
        esl_records
    )

    if language_df.empty:

        print("The file is empty.")

        continue

    language_df["_Question_Normalized"] = (
        language_df["question"]
        .apply(normalize_text)
    )

    language_df["_Subject_Normalized"] = (
        language_df["subject"]
        .apply(normalize_text)
    )

    language_df["_Choices_Signature"] = (
        language_df["choices"]
        .apply(choices_signature)
    )

    language_df["_Answer_Normalized"] = (
        language_df["answer"]
        .apply(normalize_answer)
    )

    print(
        f"Rows loaded: {len(language_df)}"
    )

    print(
        f"Subjects found: "
        f"{language_df['_Subject_Normalized'].nunique()}"
    )

    subject_assignments = {}
    subject_second_best = defaultdict(float)

    # --------------------------------------------------------
    # Match within each subject
    # --------------------------------------------------------

    for subject_number, subject in enumerate(
        language_df[
            "_Subject_Normalized"
        ].unique(),
        start=1
    ):

        esl_group = language_df[
            language_df[
                "_Subject_Normalized"
            ] == subject
        ]

        original_group = original_df[
            original_df[
                "_Subject_Normalized"
            ] == subject
        ]

        if original_group.empty:
            continue

        edges = subject_level_candidates(
            original_group,
            esl_group
        )

        group_assignments = assign_candidate_edges(
            edges
        )

        # Store the best two similarities for confidence review
        similarities_by_esl = defaultdict(list)

        for edge in edges:

            similarities_by_esl[
                edge["ESL_Index"]
            ].append(
                edge["Similarity"]
            )

        for esl_index, similarities in (
            similarities_by_esl.items()
        ):

            unique_scores = sorted(
                set(similarities),
                reverse=True
            )

            if len(unique_scores) >= 2:

                subject_second_best[
                    esl_index
                ] = unique_scores[1]

            elif unique_scores:

                subject_second_best[
                    esl_index
                ] = 0.0

        for esl_index, assignment in (
            group_assignments.items()
        ):

            # Low subject-level scores are sent to global fallback
            if (
                assignment["Similarity"]
                >= GLOBAL_FALLBACK_THRESHOLD
            ):

                subject_assignments[
                    esl_index
                ] = assignment

        if subject_number % 10 == 0:

            print(
                f"Processed "
                f"{subject_number} subject groups..."
            )

    # --------------------------------------------------------
    # Identify difficult rows
    # --------------------------------------------------------

    used_original_indices = {
        assignment["Original_Index"]
        for assignment
        in subject_assignments.values()
    }

    unresolved_indices = [
        index
        for index in language_df.index
        if index not in subject_assignments
    ]

    print(
        f"Subject-level matches: "
        f"{len(subject_assignments)}"
    )

    print(
        f"Rows sent to global fallback: "
        f"{len(unresolved_indices)}"
    )

    # --------------------------------------------------------
    # Global fallback
    # --------------------------------------------------------

    fallback_assignments = global_fallback_matches(
        language_df,
        unresolved_indices,
        used_original_indices
    )

    final_assignments = dict(
        subject_assignments
    )

    final_assignments.update(
        fallback_assignments
    )

    # --------------------------------------------------------
    # Create final output rows
    # --------------------------------------------------------

    language_output_rows = []

    for esl_index in language_df.index:

        assignment = final_assignments.get(
            esl_index
        )

        second_best = subject_second_best.get(
            esl_index,
            0.0
        )

        output_row = build_output_row(
            language,
            language_df,
            esl_index,
            assignment,
            second_best
        )

        language_output_rows.append(
            output_row
        )

    language_output_df = pd.DataFrame(
        language_output_rows
    )

    language_output_file = os.path.join(
        OUTPUT_DIR,
        (
            f"mmlu_{language.lower()}_"
            "fast_match_audit.csv"
        )
    )

    language_output_df.to_csv(
        language_output_file,
        index=False,
        encoding="utf-8-sig"
    )

    all_language_outputs.append(
        language_output_df
    )

    matched_count = int(
        (
            language_output_df[
                "Matched_Original_ID"
            ].astype(str).str.len() > 0
        ).sum()
    )

    high_count = int(
        (
            language_output_df[
                "Match_Status"
            ] == "HIGH"
        ).sum()
    )

    review_count = int(
        (
            language_output_df[
                "Match_Status"
            ] == "REVIEW"
        ).sum()
    )

    low_count = int(
        (
            language_output_df[
                "Match_Status"
            ] == "LOW"
        ).sum()
    )

    integrity_ok_count = int(
        (
            language_output_df[
                "Integrity_Status"
            ] == "OK"
        ).sum()
    )

    metadata_problem_count = int(
        (
            language_output_df[
                "Integrity_Status"
            ] != "OK"
        ).sum()
    )

    unmatched_count = int(
        (
            language_output_df[
                "Integrity_Status"
            ] == "UNMATCHED"
        ).sum()
    )

    summary_rows.append(
        {
            "Language": language,
            "Input_Rows": len(
                language_output_df
            ),
            "Matched_Rows": matched_count,
            "High_Confidence": high_count,
            "Needs_Review": review_count,
            "Low_Confidence": low_count,
            "Integrity_OK": integrity_ok_count,
            "Metadata_Problems": metadata_problem_count,
            "Unmatched": unmatched_count,
            "JSON_Errors": len(
                esl_errors
            ),
            "File_Status": "OK",
        }
    )

    print(
        f"Finished {language}: "
        f"{matched_count}/{len(language_output_df)} matched"
    )

    print(
        f"High={high_count}, "
        f"Review={review_count}, "
        f"Low={low_count}"
    )

    print(
        f"Integrity OK={integrity_ok_count}, "
        f"Metadata problems={metadata_problem_count}"
    )

    print()


# ============================================================
# 11. Combine and save all outputs
# ============================================================

if not all_language_outputs:

    raise ValueError(
        "No language files were processed."
    )


combined_df = pd.concat(
    all_language_outputs,
    ignore_index=True
)

summary_df = pd.DataFrame(
    summary_rows
)


combined_csv = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_fast_match_all.csv"
)

combined_jsonl = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_fast_match_all.jsonl"
)

summary_csv = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_fast_match_summary.csv"
)

review_csv = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_needs_review.csv"
)

integrity_csv = os.path.join(
    OUTPUT_DIR,
    "mmlu_esl_integrity_problems.csv"
)


combined_df.to_csv(
    combined_csv,
    index=False,
    encoding="utf-8-sig"
)

combined_df.to_json(
    combined_jsonl,
    orient="records",
    lines=True,
    force_ascii=False
)

summary_df.to_csv(
    summary_csv,
    index=False,
    encoding="utf-8-sig"
)


review_df = combined_df[
    combined_df[
        "Match_Status"
    ].isin(
        [
            "REVIEW",
            "LOW",
        ]
    )
].copy()

review_df.to_csv(
    review_csv,
    index=False,
    encoding="utf-8-sig"
)


integrity_df = combined_df[
    combined_df[
        "Integrity_Status"
    ] != "OK"
].copy()

integrity_df.to_csv(
    integrity_csv,
    index=False,
    encoding="utf-8-sig"
)


# ============================================================
# 12. Final report
# ============================================================

print("=" * 70)
print("MMLU FAST MATCHING SUMMARY")
print("=" * 70)

display(summary_df)

print()

print(
    f"Total ESL rows: "
    f"{len(combined_df)}"
)

print(
    f"Total matched rows: "
    f"{combined_df['Matched_Original_ID'].astype(str).str.len().gt(0).sum()}"
)

print(
    f"High-confidence matches: "
    f"{(combined_df['Match_Status'] == 'HIGH').sum()}"
)

print(
    f"Review matches: "
    f"{(combined_df['Match_Status'] == 'REVIEW').sum()}"
)

print(
    f"Low-confidence matches: "
    f"{(combined_df['Match_Status'] == 'LOW').sum()}"
)

print(
    f"Integrity problems: "
    f"{len(integrity_df)}"
)

print()

print(
    f"Combined file: {combined_csv}"
)

print(
    f"Summary file: {summary_csv}"
)

print(
    f"Review file: {review_csv}"
)

print(
    f"Integrity file: {integrity_csv}"
)


# ============================================================
# 13. Zip and download
# ============================================================

zip_file = shutil.make_archive(
    ZIP_BASE,
    "zip",
    OUTPUT_DIR
)

print()
print(f"ZIP saved: {zip_file}")

files.download(
    zip_file
)

ORIGINAL REFERENCE
Original records loaded: 14042
Original subjects: 57

BUILDING GLOBAL FALLBACK INDEX
This is built once and is only used for difficult or incorrectly labelled rows.
Global fallback index ready.

PROCESSING: Arabic
Rows loaded: 7246
Subjects found: 54
Processed 10 subject groups...
Processed 20 subject groups...
Processed 30 subject groups...
Processed 40 subject groups...
Processed 50 subject groups...
Subject-level matches: 5082
Rows sent to global fallback: 2164
Finished Arabic: 7201/7246 matched
High=2655, Review=2294, Low=2297
Integrity OK=5678, Metadata problems=1568

PROCESSING: Chinese
Rows loaded: 7246
Subjects found: 54
Processed 10 subject groups...
Processed 20 subject groups...
Processed 30 subject groups...
Processed 40 subject groups...
Processed 50 subject groups...
Subject-level matches: 5423
Rows sent to global fallback: 1823
Finished Chinese: 7189/7246 matched
High=2665, Review=2302, Low=2279
Integrity OK=6033, Metadata problems=1213

PROCESSING: Fr

,Language,Input_Rows,Matched_Rows,High_Confidence,Needs_Review,Low_Confidence,Integrity_OK,Metadata_Problems,Unmatched,JSON_Errors,File_Status
0,Arabic,7246,7201,2655,2294,2297,5678,1568,45,0,OK
1,Chinese,7246,7189,2665,2302,2279,6033,1213,57,0,OK
2,French,7246,7188,2823,2242,2181,6437,809,58,0,OK
3,German,7246,7196,2614,2314,2318,6040,1206,50,0,OK
4,Japanese,7246,7196,2576,2289,2381,6006,1240,50,0,OK
5,Portuguese,7246,7190,2569,2331,2346,5807,1439,56,0,OK
6,Russian,7246,7195,2646,2283,2317,5891,1355,51,0,OK
7,Spanish,7246,7206,2569,2324,2353,5735,1511,40,0,OK



Total ESL rows: 57968
Total matched rows: 57561
High-confidence matches: 21117
Review matches: 18379
Low-confidence matches: 18472
Integrity problems: 10341

Combined file: /content/mmlu_fast_match_output/mmlu_esl_fast_match_all.csv
Summary file: /content/mmlu_fast_match_output/mmlu_esl_fast_match_summary.csv
Review file: /content/mmlu_fast_match_output/mmlu_esl_needs_review.csv
Integrity file: /content/mmlu_fast_match_output/mmlu_esl_integrity_problems.csv

ZIP saved: /content/mmlu_fast_match_output.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [19]:
import pandas as pd
from google.colab import files


# ==========================================
# Load combined matching result
# ==========================================

INPUT_FILE = (
    "/content/mmlu_fast_match_output/"
    "mmlu_esl_fast_match_all.csv"
)

OUTPUT_FILE = (
    "/content/mmlu_strict_shared_candidates.csv"
)

df = pd.read_csv(INPUT_FILE)


# ==========================================
# Keep only strict reliable matches
# ==========================================

strict_df = df[
    (df["Match_Status"] == "HIGH")
    & (df["Integrity_Status"] == "OK")
    & (df["Matched_Original_ID"].notna())
    & (df["Matched_Original_ID"].astype(str).str.strip() != "")
].copy()


print("=" * 70)
print("STRICT MATCHES BY LANGUAGE")
print("=" * 70)

strict_counts = (
    strict_df
    .groupby("Language")
    .size()
    .reset_index(name="Strict_Rows")
)

display(strict_counts)


# ==========================================
# Find original IDs appearing in all 8 languages
# ==========================================

id_summary = (
    strict_df
    .groupby("Matched_Original_ID")
    .agg(
        Row_Count=("Language", "size"),
        Language_Count=("Language", "nunique")
    )
    .reset_index()
)


valid_ids = id_summary[
    (id_summary["Row_Count"] == 8)
    & (id_summary["Language_Count"] == 8)
]["Matched_Original_ID"]


shared_df = strict_df[
    strict_df["Matched_Original_ID"].isin(valid_ids)
].copy()


shared_df = shared_df.sort_values(
    by=[
        "Matched_Original_ID",
        "Language"
    ]
)


# ==========================================
# Report
# ==========================================

print()
print("=" * 70)
print("SHARED STRICT ORIGINAL PROMPTS")
print("=" * 70)

print(
    "Strict high-confidence rows: "
    f"{len(strict_df)}"
)

print(
    "Original prompts represented in all "
    f"8 languages: {len(valid_ids)}"
)

print(
    "Total rows in shared candidate pool: "
    f"{len(shared_df)}"
)

print()

if len(valid_ids) >= 10:
    print(
        "Result: enough reliable shared prompts "
        "for sampling 10 base prompts."
    )
else:
    print(
        "Result: fewer than 10 reliable shared prompts. "
        "Do not sample yet."
    )


# ==========================================
# Save candidate pool
# ==========================================

shared_df.to_csv(
    OUTPUT_FILE,
    index=False,
    encoding="utf-8-sig"
)

print()
print(f"Saved: {OUTPUT_FILE}")

display(shared_df.head(16))

files.download(OUTPUT_FILE)

STRICT MATCHES BY LANGUAGE


,Language,Strict_Rows
0,Arabic,2197
1,Chinese,2353
2,French,2638
3,German,2328
4,Japanese,2285
5,Portuguese,2172
6,Russian,2287
7,Spanish,2157



SHARED STRICT ORIGINAL PROMPTS
Strict high-confidence rows: 18417
Original prompts represented in all 8 languages: 949
Total rows in shared candidate pool: 7592

Result: enough reliable shared prompts for sampling 10 base prompts.

Saved: /content/mmlu_strict_shared_candidates.csv


,Language,ESL_Row,ESL_Subject,ESL_Question,ESL_Choices,ESL_Answer,Matched_Original_ID,Original_Row,Original_Subject,Original_Prompt,...,Gold_Answer,Question_Similarity,Second_Best_Similarity,Similarity_Margin,Candidate_Rank,Subject_Preserved,Choices_Preserved,Answer_Preserved,Match_Status,Integrity_Status
2,Arabic,3,abstract_algebra,"""Find the product given polynomial in Z_8[x]. ...","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9189,0.8327,0.0862,1.0,Yes,Yes,Yes,HIGH,OK
7248,Chinese,3,abstract_algebra,"""Find the product of given polynomials in Z_8[...","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9425,0.8595,0.0830,1.0,Yes,Yes,Yes,HIGH,OK
14494,French,3,abstract_algebra,"""Find the product given polynomials in Z_8[x]....","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9414,0.8574,0.0840,1.0,Yes,Yes,Yes,HIGH,OK
21740,German,3,abstract_algebra,"""Find the product given polynomials in Z_8[x]....","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9410,0.8577,0.0833,1.0,Yes,Yes,Yes,HIGH,OK
28986,Japanese,3,abstract_algebra,"""Find the product given polynomials in Z_8[x]....","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9417,0.8569,0.0847,1.0,Yes,Yes,Yes,HIGH,OK
36232,Portuguese,3,abstract_algebra,"""Find the product given polynomials in Z_8[x] ...","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9023,0.8285,0.0739,1.0,Yes,Yes,Yes,HIGH,OK
43478,Russian,3,abstract_algebra,Finds of the product of given polynomials in Z...,"[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.9160,0.8349,0.0811,1.0,Yes,Yes,Yes,HIGH,OK
50724,Spanish,3,abstract_algebra,"""Find locate a product given polynomials in Z_...","[""2x^2 + 5"", ""6x^2 + 4x + 6"", ""0"", ""x^2 + 1""]",1,MMLU_00005,5.0,abstract_algebra,Find the product of the given polynomials in t...,...,1.0,0.8738,0.7950,0.0788,1.0,Yes,Yes,Yes,HIGH,OK
6,Arabic,7,abstract_algebra,"""Finds the order the factor group (Z_11 x Z_15...","[""1"", ""2"", ""5"", ""11""]",0,MMLU_00013,13.0,abstract_algebra,Find the order of the factor group (Z_11 x Z_1...,...,0.0,0.9667,0.1896,0.7771,1.0,Yes,Yes,Yes,HIGH,OK
7252,Chinese,7,abstract_algebra,"""Finds the order of the factor group (Z_11 x Z...","[""1"", ""2"", ""5"", ""11""]",0,MMLU_00013,13.0,abstract_algebra,Find the order of the factor group (Z_11 x Z_1...,...,0.0,0.9606,0.1879,0.7727,1.0,Yes,Yes,Yes,HIGH,OK


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
import json
import pandas as pd
from google.colab import files


# ============================================================
# Settings
# ============================================================

INPUT_FILE = (
    "/content/mmlu_strict_shared_candidates.csv"
)

OUTPUT_CSV = (
    "/content/mmlu_human_validation_sample_80.csv"
)

OUTPUT_JSONL = (
    "/content/mmlu_human_validation_sample_80.jsonl"
)

REVIEWER_CSV = (
    "/content/mmlu_human_validation_reviewer_sheet.csv"
)

RANDOM_SEED = 42
NUMBER_OF_BASE_PROMPTS = 10


LANGUAGE_ORDER = [
    "Arabic",
    "Chinese",
    "French",
    "German",
    "Japanese",
    "Portuguese",
    "Russian",
    "Spanish",
]


# ============================================================
# Load strict shared candidate pool
# ============================================================

df = pd.read_csv(INPUT_FILE)

print("=" * 70)
print("MMLU STRICT SHARED CANDIDATE POOL")
print("=" * 70)

print(f"Input rows: {len(df)}")

print(
    "Unique original prompts: "
    f"{df['Matched_Original_ID'].nunique()}"
)

print(
    "Languages: "
    f"{df['Language'].nunique()}"
)


# ============================================================
# Final eligibility check
# ============================================================

group_check = (
    df.groupby("Matched_Original_ID")
    .agg(
        Row_Count=("Language", "size"),
        Language_Count=("Language", "nunique"),
        Match_Status_Count=("Match_Status", "nunique"),
        Integrity_Status_Count=(
            "Integrity_Status",
            "nunique"
        ),
    )
    .reset_index()
)


valid_ids = group_check[
    (group_check["Row_Count"] == 8)
    & (group_check["Language_Count"] == 8)
]["Matched_Original_ID"]


eligible_df = df[
    df["Matched_Original_ID"].isin(valid_ids)
].copy()


# Additional strict checks
eligible_df = eligible_df[
    (eligible_df["Match_Status"] == "HIGH")
    & (eligible_df["Integrity_Status"] == "OK")
].copy()


print()
print(
    "Eligible original prompts after final check: "
    f"{eligible_df['Matched_Original_ID'].nunique()}"
)


if (
    eligible_df["Matched_Original_ID"].nunique()
    < NUMBER_OF_BASE_PROMPTS
):
    raise ValueError(
        "There are not enough eligible original prompts."
    )


# ============================================================
# Randomly sample 10 unique original IDs
# ============================================================

original_id_table = (
    eligible_df[
        [
            "Matched_Original_ID",
            "Original_Subject",
            "Original_Prompt",
        ]
    ]
    .drop_duplicates(
        subset=["Matched_Original_ID"]
    )
    .reset_index(drop=True)
)


sampled_id_table = (
    original_id_table.sample(
        n=NUMBER_OF_BASE_PROMPTS,
        random_state=RANDOM_SEED,
        replace=False,
    )
    .reset_index(drop=True)
)


sampled_id_table["Sample_ID"] = [
    f"MMLU_SAMPLE_{number:02d}"
    for number in range(
        1,
        NUMBER_OF_BASE_PROMPTS + 1
    )
]


# ============================================================
# Retrieve all eight languages for sampled IDs
# ============================================================

sample_df = eligible_df.merge(
    sampled_id_table[
        [
            "Matched_Original_ID",
            "Sample_ID",
        ]
    ],
    on="Matched_Original_ID",
    how="inner",
)


# Order languages consistently
sample_df["Language"] = pd.Categorical(
    sample_df["Language"],
    categories=LANGUAGE_ORDER,
    ordered=True,
)


sample_df = sample_df.sort_values(
    by=[
        "Sample_ID",
        "Language",
    ]
).reset_index(drop=True)


# ============================================================
# Select and rename final columns
# ============================================================

final_columns = [
    "Sample_ID",
    "Matched_Original_ID",
    "Original_Row",
    "Original_Subject",
    "Language",
    "Original_Prompt",
    "ESL_Question",
    "Original_Choices",
    "ESL_Choices",
    "Gold_Answer",
    "ESL_Answer",
    "Question_Similarity",
    "Match_Status",
    "Integrity_Status",
]


sample_df = sample_df[
    final_columns
].copy()


sample_df = sample_df.rename(
    columns={
        "Matched_Original_ID": "Original_ID",
        "Original_Subject": "Subject",
    }
)


# ============================================================
# Validate final sample
# ============================================================

sample_group_check = (
    sample_df.groupby("Original_ID")
    .agg(
        Row_Count=("Language", "size"),
        Language_Count=("Language", "nunique"),
    )
)


language_counts = (
    sample_df["Language"]
    .value_counts(sort=False)
    .reset_index()
)

language_counts.columns = [
    "Language",
    "Sample_Count",
]


print()
print("=" * 70)
print("FINAL SAMPLE CHECK")
print("=" * 70)

print(
    f"Unique base prompts: "
    f"{sample_df['Original_ID'].nunique()}"
)

print(
    f"Total ESL rows: "
    f"{len(sample_df)}"
)

print(
    "Every original has exactly 8 rows: "
    f"{bool((sample_group_check['Row_Count'] == 8).all())}"
)

print(
    "Every original has exactly 8 languages: "
    f"{bool((sample_group_check['Language_Count'] == 8).all())}"
)

print()
display(language_counts)


if len(sample_df) != 80:
    raise ValueError(
        f"Expected 80 rows, but obtained {len(sample_df)}."
    )


if not (
    sample_group_check["Row_Count"] == 8
).all():
    raise ValueError(
        "At least one original prompt does not have 8 rows."
    )


if not (
    sample_group_check["Language_Count"] == 8
).all():
    raise ValueError(
        "At least one original prompt does not have 8 languages."
    )


# ============================================================
# Save analysis sample
# ============================================================

sample_df.to_csv(
    OUTPUT_CSV,
    index=False,
    encoding="utf-8-sig",
)

sample_df.to_json(
    OUTPUT_JSONL,
    orient="records",
    lines=True,
    force_ascii=False,
)


# ============================================================
# Create human-review sheet
# ============================================================

reviewer_df = sample_df.copy()


review_columns = [
    "R1_Meaning",
    "R2_Meaning",
    "Final_Meaning",

    "R1_Key_Info",
    "R2_Key_Info",
    "Final_Key_Info",

    "R1_Realism",
    "R2_Realism",
    "Final_Realism",

    "R1_Readability",
    "R2_Readability",
    "Final_Readability",

    "Comments",
]


for column in review_columns:
    reviewer_df[column] = ""


reviewer_df.to_csv(
    REVIEWER_CSV,
    index=False,
    encoding="utf-8-sig",
)


# ============================================================
# Display selected prompts
# ============================================================

print()
print("=" * 70)
print("SELECTED ORIGINAL PROMPTS")
print("=" * 70)

display(
    sample_df[
        [
            "Sample_ID",
            "Original_ID",
            "Subject",
            "Original_Prompt",
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print()
print(f"Sample CSV saved: {OUTPUT_CSV}")
print(f"Sample JSONL saved: {OUTPUT_JSONL}")
print(f"Reviewer sheet saved: {REVIEWER_CSV}")


# ============================================================
# Download
# ============================================================

files.download(OUTPUT_CSV)
files.download(OUTPUT_JSONL)
files.download(REVIEWER_CSV)

MMLU STRICT SHARED CANDIDATE POOL
Input rows: 7592
Unique original prompts: 949
Languages: 8

Eligible original prompts after final check: 949

FINAL SAMPLE CHECK
Unique base prompts: 10
Total ESL rows: 80
Every original has exactly 8 rows: True
Every original has exactly 8 languages: True



,Language,Sample_Count
0,Arabic,10
1,Chinese,10
2,French,10
3,German,10
4,Japanese,10
5,Portuguese,10
6,Russian,10
7,Spanish,10



SELECTED ORIGINAL PROMPTS


,Sample_ID,Original_ID,Subject,Original_Prompt
0,MMLU_SAMPLE_01,MMLU_02429,elementary_mathematics,What is the value of the expression 28 x 42?
1,MMLU_SAMPLE_02,MMLU_07952,miscellaneous,According to a Yale University study what smel...
2,MMLU_SAMPLE_03,MMLU_08066,miscellaneous,What day of the week is sometimes called 'hump...
3,MMLU_SAMPLE_04,MMLU_08593,moral_scenarios,For which of these two scenarios does the main...
4,MMLU_SAMPLE_05,MMLU_09421,nutrition,What role do women play in food security?
5,MMLU_SAMPLE_06,MMLU_09070,moral_scenarios,For which of these two scenarios does the main...
6,MMLU_SAMPLE_07,MMLU_10421,professional_accounting,Johnson worked for ABC Co. and earned a salary...
7,MMLU_SAMPLE_08,MMLU_13778,virology,The largest Latino community in the U.S. is:
8,MMLU_SAMPLE_09,MMLU_06678,logical_fallacies,Arguing that what is true of an entire object ...
9,MMLU_SAMPLE_10,MMLU_02337,elementary_mathematics,Which measurement would best be rounded to the...



Sample CSV saved: /content/mmlu_human_validation_sample_80.csv
Sample JSONL saved: /content/mmlu_human_validation_sample_80.jsonl
Reviewer sheet saved: /content/mmlu_human_validation_reviewer_sheet.csv


...

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>